# Step 5 — Talk DNA Pattern Detector
**Tough Talks · Phase 3**

Goal: prove Gemma 4 E2B (text-only) can read a conversation transcript
(optionally enriched with Step 4 emotion data) and emit a TalkDNA
profile matching `data/schemas/talk_dna.schema.json`.

**Hybrid design** — the precision-sensitive numerics are computed in
code (`apology_rate`, `avg_turn_length_words`, filler candidates,
`interruption_rate` when timing is available); the qualitative fields
(`sarcasm_frequency`, `silence_under_pressure`, `escalation_triggers`,
curated `filler_phrases`, `strengths`, `weaknesses`) come from a single
prompt-based call. Same two-shot retry pattern as Step 04 (greedy,
then light sampling on JSON / validation failure).

Notebook is a thin driver; all logic lives in
`backend/core/_runtime/talk_dna.py`.

**What "done" looks like for this step**
1. Text-only Gemma 4 loads (`AutoModelForCausalLM` via `LoadConfig(multimodal=False)`).
2. `compute_deterministic_metrics()` returns sensible numerics on both sample conversations.
3. `analyze_talk_dna()` returns a TalkDNA dict for conversation A (`version=1`).
4. `analyze_talk_dna()` with `prior_profile=profile_v1` returns an updated dict for conversation B (`version=2`, `conversation_count=2`, numerics weighted-averaged across both conversations).
5. Every result validates against `data/schemas/talk_dna.schema.json` (required fields, sarcasm enum, identifier shape).

**Why text-only**
TalkDNA reasons over WORDS — apologies, hedges, sarcasm, what triggered
escalation. Prosody is captured upstream by Step 04 and surfaces here
as the optional `emotion` field on each turn, which the prompt renders
as part of the transcript. No new audio path is needed.

In [ ]:
# ── 0. Install / upgrade dependencies ─────────────────────────────────
# Text-only path — no audio libs required. Same rule as the audio steps:
# bump only transformers + accelerate on Colab / Kaggle (bumping torch
# breaks the pre-installed torchvision / CUDA pairing). After this first
# run, RESTART THE KERNEL before continuing if you actually upgraded
# transformers — the already-imported version won't pick up the change.

!pip install -q -U transformers accelerate

In [ ]:
# ── 1. Locate (or fetch) the repo, put it on sys.path ─────────────────────
# Same shim as Steps 01–04 — auto-clones / refreshes on Colab / Kaggle.
# After the refresh we drop any cached `backend.*` modules so subsequent
# imports pick up the freshly-pulled code instead of whatever this kernel
# imported earlier in the session.

import os, pathlib, subprocess, sys

REPO_URL  = "https://github.com/EhsanFarazmand/tough_talks.git"
REPO_NAME = "tough_talks"

def _looks_like_repo(p: pathlib.Path) -> bool:
    return (p / "backend" / "core" / "_runtime").is_dir()

def _scan_for_repo() -> pathlib.Path | None:
    cwd = pathlib.Path.cwd()
    for parent in [cwd, *cwd.parents]:
        if _looks_like_repo(parent):
            return parent
    for base in (pathlib.Path("/content"), pathlib.Path("/kaggle/working")):
        candidate = base / REPO_NAME
        if _looks_like_repo(candidate):
            return candidate
    return None

def _refresh(target: pathlib.Path) -> None:
    if not (target / ".git").is_dir():
        return
    print(f"Refreshing {target} from origin")
    subprocess.run(["git", "-C", str(target), "fetch", "--depth", "1", "origin"],
                   capture_output=True, check=False)
    subprocess.run(["git", "-C", str(target), "reset", "--hard", "FETCH_HEAD"],
                   capture_output=True, check=False)

REPO_ROOT = _scan_for_repo()
if REPO_ROOT is None:
    base = next((b for b in (pathlib.Path("/content"), pathlib.Path("/kaggle/working")) if b.is_dir()),
                pathlib.Path.cwd())
    target = base / REPO_NAME
    print(f"Cloning {REPO_URL} -> {target}")
    result = subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(target)],
                            capture_output=True, text=True)
    if result.returncode != 0:
        raise RuntimeError("git clone failed:\n" + result.stderr)
    REPO_ROOT = target
else:
    _refresh(REPO_ROOT)

os.chdir(REPO_ROOT)
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

_stale = [m for m in list(sys.modules) if m == "backend" or m.startswith("backend.")]
for _m in _stale:
    del sys.modules[_m]
if _stale:
    print(f"Cleared {len(_stale)} cached backend.* module(s) from sys.modules")

print(f"Repo root: {REPO_ROOT}")

In [ ]:
# ── 2. Imports ──────────────────────────────────────────────────
import json
from pathlib import Path

import torch

from backend.core._runtime import (
    ALLOWED_SARCASM_FREQUENCIES,
    DEFAULT_MODEL_ID,
    JsonParseError,
    LoadConfig,
    TalkDNAAnalysisError,
    TalkDNAConfig,
    analyze_talk_dna,
    compute_deterministic_metrics,
    load_model,
)

In [ ]:
# ── 3. Configuration ─────────────────────────────────────────────────

MODEL_ID    = DEFAULT_MODEL_ID                # google/gemma-4-E2B-it
DEVICE      = "cuda" if torch.cuda.is_available() else "cpu"
SCHEMA_PATH = Path(REPO_ROOT) / "data" / "schemas" / "talk_dna.schema.json"

# Conversation A — same nine-turn argument transcript as Steps 03 / 04, in
# text form, with the per-turn emotion readings Step 04 produced. The user
# over-apologises, hedges ("I just", "kind of"), but eventually concedes.
# This is the source of the v1 TalkDNA profile.
CONVERSATION_A = [
    {"speaker": "user",  "text": "I just wanted to check on the report — it was due Monday and it's already Thursday.",
     "emotion": {"primary": "frustration", "intensity": 0.7}},
    {"speaker": "other", "text": "I told you on Tuesday the data team hadn't delivered. I can't just make numbers up.",
     "emotion": {"primary": "defensiveness", "intensity": 0.8}},
    {"speaker": "user",  "text": "Sorry, I kind of forgot you mentioned that. But you still should escalate — you don't just sit on it.",
     "emotion": {"primary": "frustration", "intensity": 0.75}},
    {"speaker": "other", "text": "I did escalate. You weren't in the meeting. Don't blame me for your missed message.",
     "emotion": {"primary": "defensiveness", "intensity": 0.75}},
    {"speaker": "user",  "text": "I guess I just can't be in every meeting. That's why we have email — sorry, I should've checked it.",
     "emotion": {"primary": "frustration", "intensity": 0.6}},
    {"speaker": "other", "text": "I sent two emails. You replied to neither. Don't put this on me.",
     "emotion": {"primary": "frustration", "intensity": 0.7}},
    {"speaker": "user",  "text": "Okay, fair. My bad, I missed them. But we still have a problem to solve tonight.",
     "emotion": {"primary": "frustration", "intensity": 0.55, "concession_made": True}},
    {"speaker": "other", "text": "I have partial numbers from the staging tables. We can present those and flag the gaps.",
     "emotion": {"primary": "openness", "intensity": 0.5}},
    {"speaker": "user",  "text": "Good. Let's regroup at six. And next time, just call me directly — I'm sorry I missed the emails.",
     "emotion": {"primary": "neutral", "intensity": 0.3}},
]

# Conversation B — a shorter follow-up the next morning. The user is
# slightly clearer (fewer apologies, less hedging) but the same underlying
# pattern shows. This drives the v2 profile via the incremental path.
CONVERSATION_B = [
    {"speaker": "user",  "text": "I want to set a hard deadline for the data team — Wednesday EOD, no exceptions."},
    {"speaker": "other", "text": "That's aggressive given the staging tables aren't done."},
    {"speaker": "user",  "text": "Sorry, I know it's tight. But I just need a number we can defend to finance."},
    {"speaker": "other", "text": "Then commit to escalating Tuesday morning if blockers are still open."},
    {"speaker": "user",  "text": "Fair. I'll send the calendar block today."},
]

print(f"Model    : {MODEL_ID}")
print(f"Device   : {DEVICE}")
print(f"Conv A   : {len(CONVERSATION_A)} turns")
print(f"Conv B   : {len(CONVERSATION_B)} turns")
print(f"Sarcasm  : {ALLOWED_SARCASM_FREQUENCIES}")

In [ ]:
# ── 4. Deterministic metrics sanity-check (no model needed) ────────────────
# `compute_deterministic_metrics` is pure Python — we can inspect what the
# code-side accounting looks like before paying the model-loading cost.
# This block is also useful for debugging the regex/filler detection on
# any new transcript.

for label, turns in (("A", CONVERSATION_A), ("B", CONVERSATION_B)):
    m = compute_deterministic_metrics(turns)
    print(f"Conversation {label}:")
    print(f"  user_turn_count        : {m.user_turn_count}")
    print(f"  apology_rate           : {m.apology_rate:.2f}")
    print(f"  avg_user_turn_words    : {m.avg_user_turn_words:.1f}")
    print(f"  filler_candidates      : {m.filler_candidates}")
    print(f"  interruption_rate      : {m.interruption_rate}")

In [ ]:
# ── 5. Load the text-only processor + model ────────────────────────────
# TalkDNA reasons over WORDS — we don't need the multimodal audio path
# here. `multimodal=False` (the default) loads `AutoModelForCausalLM`,
# which is lighter on VRAM and slightly faster than the multimodal class.

processor, model = load_model(LoadConfig(model_id=MODEL_ID))
n_params = sum(p.numel() for p in model.parameters()) / 1e9
print(f"Model loaded ({n_params:.1f}B parameters, on {model.device})")

In [ ]:
# ── 6. Conversation A → first TalkDNA profile (version 1) ─────────────────
# The runtime renders the prompt from (a) the transcript with emotion
# tags, (b) the deterministic metrics block, and (c) a "no prior"
# sentinel for the prior-profile block. Greedy first, light sampling on
# parse failure (same retry policy as Step 04).

profile_v1 = analyze_talk_dna(
    processor,
    model,
    CONVERSATION_A,
    cfg=TalkDNAConfig(user_id="local"),
)
print(json.dumps(profile_v1, indent=2))

In [ ]:
# ── 7. Conversation B with prior_profile=v1 → updated profile (version 2) ───
# This is the incremental path. The runtime bumps `version` to 2,
# increments `conversation_count`, and weighted-averages the
# deterministic numerics (apology_rate, avg_user_turn_words) against the
# v1 values. The qualitative fields evolve via the prompt, which now
# sees the v1 profile as prior context.

profile_v2 = analyze_talk_dna(
    processor,
    model,
    CONVERSATION_B,
    cfg=TalkDNAConfig(user_id="local", prior_profile=profile_v1),
)
print(json.dumps(profile_v2, indent=2))

In [ ]:
# ── 8. Schema validation + results table ───────────────────────────────
# Hand-rolled validator (no jsonschema dep — matches Step 04). Per profile:
#   - required top-level fields present
#   - patterns.sarcasm_frequency in the schema enum
#   - apology_rate / avg_turn_length_words / interruption_rate (when present) numeric
#   - strengths / weaknesses are snake_case identifiers

import re

schema = json.loads(SCHEMA_PATH.read_text(encoding="utf-8"))
SCHEMA_REQUIRED  = schema["required"]
SARCASM_ENUM     = schema["properties"]["patterns"]["properties"]["sarcasm_frequency"]["enum"]
IDENT_RE         = re.compile(r"^[a-z][a-z0-9_]*$")


def _validate_talk_dna(profile: dict) -> list[str]:
    errs: list[str] = []
    for k in SCHEMA_REQUIRED:
        if k not in profile:
            errs.append(f"missing top-level key {k!r}")
    patterns = profile.get("patterns")
    if not isinstance(patterns, dict):
        errs.append("patterns is not an object")
        return errs
    sarcasm = patterns.get("sarcasm_frequency")
    if sarcasm is not None and sarcasm not in SARCASM_ENUM:
        errs.append(f"sarcasm_frequency {sarcasm!r} not in {SARCASM_ENUM}")
    for nk in ("apology_rate", "avg_turn_length_words", "interruption_rate"):
        v = patterns.get(nk)
        if v is None:
            continue
        if isinstance(v, bool) or not isinstance(v, (int, float)):
            errs.append(f"patterns.{nk} not numeric: {v!r}")
        elif nk in ("apology_rate", "interruption_rate") and not 0.0 <= float(v) <= 1.0:
            errs.append(f"patterns.{nk} out of range: {v}")
    for k in ("strengths", "weaknesses"):
        items = profile.get(k, [])
        if not isinstance(items, list):
            errs.append(f"{k} is not a list")
            continue
        for item in items:
            if not isinstance(item, str) or not IDENT_RE.match(item):
                errs.append(f"{k} item not a snake_case identifier: {item!r}")
    return errs


profiles = [("v1 (conversation A)", profile_v1), ("v2 (after conversation B)", profile_v2)]
per_profile_errors = [(label, _validate_talk_dna(p)) for label, p in profiles]
n_clean = sum(1 for _, errs in per_profile_errors if not errs)

checks: list[tuple[str, bool, str]] = [
    ("text_model_loaded",       True,                              f"{n_params:.1f}B params on {model.device}"),
    ("profile_v1_built",        isinstance(profile_v1, dict),       f"version={profile_v1.get('version')}"),
    ("profile_v2_built",        isinstance(profile_v2, dict),       f"version={profile_v2.get('version')}"),
    ("version_incremented",     profile_v2.get("version") == 2,     f"v1={profile_v1.get('version')} -> v2={profile_v2.get('version')}"),
    ("conversation_count_2",    profile_v2.get("conversation_count") == 2,
                                                                     f"{profile_v2.get('conversation_count')}"),
    ("schema_valid_per_profile", n_clean == len(profiles),           f"{n_clean}/{len(profiles)} clean"),
    ("sarcasm_in_enum",          all(p.get("patterns", {}).get("sarcasm_frequency") in SARCASM_ENUM for _, p in profiles),
                                                                     f"v1={profile_v1['patterns']['sarcasm_frequency']!r}, v2={profile_v2['patterns']['sarcasm_frequency']!r}"),
]

print("=" * 72)
print("STEP 5 RESULTS — Gemma 4 Talk DNA pattern detector")
print("=" * 72)
all_ok = True
for name, ok, note in checks:
    icon = "PASS" if ok else "FAIL"
    print(f"[{icon}]  {name:28s}  {note}")
    if not ok:
        all_ok = False

print("\nProfile evolution:")
for label, p in profiles:
    pat = p.get("patterns", {})
    print(f"  {label}")
    print(f"    apology_rate          : {pat.get('apology_rate')}")
    print(f"    avg_turn_length_words : {pat.get('avg_turn_length_words')}")
    print(f"    sarcasm_frequency     : {pat.get('sarcasm_frequency')}")
    print(f"    silence_under_pressure: {pat.get('silence_under_pressure')}")
    print(f"    filler_phrases        : {pat.get('filler_phrases')}")
    print(f"    escalation_triggers   : {pat.get('escalation_triggers')}")
    print(f"    strengths             : {p.get('strengths')}")
    print(f"    weaknesses            : {p.get('weaknesses')}")

if any(errs for _, errs in per_profile_errors):
    print("\nSchema errors:")
    for label, errs in per_profile_errors:
        if not errs:
            continue
        print(f"  {label}:")
        for line in errs:
            print(f"    - {line}")

print()
print("OVERALL:", "READY FOR STEP 6" if all_ok else "FIX FAILURES ABOVE")